# Recursa — reproducible experiments
Upload `recursa-source.zip` to `MyDrive/Recursa/` once, then Run all. This notebook uses CPU only, mounts Drive interactively, preserves checkpoints, and exports Studio-compatible JSON/CSV/ZIP. No API keys or live inference are used. GPU inference and fine-tuning remain future work. Every new VM reconstructs the same source from durable storage; no manual code copying or package installation is needed.
Source archive: run `python scripts/package_source.py` locally. Review the archive before uploading; the packager excludes credentials and local results.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import sys, zipfile, subprocess
assert sys.version_info >= (3, 11), 'Python 3.11+ is required'
ARCHIVE = Path('/content/drive/MyDrive/Recursa/recursa-source.zip')
OUTPUT = Path('/content/drive/MyDrive/Recursa/results')
CONFIG = 'configs/smoke.json'
WORKSPACE = Path('/content/recursa-workspace')
assert ARCHIVE.is_file(), f'Place the reviewed source ZIP at {ARCHIVE}'
# A fresh source directory avoids mixing files from different archive revisions.
import tempfile
WORKSPACE.mkdir(parents=True, exist_ok=True)
SOURCE = Path(tempfile.mkdtemp(prefix='source-', dir=WORKSPACE))
with zipfile.ZipFile(ARCHIVE) as bundle:
    assert sum(item.file_size for item in bundle.infolist()) < 50_000_000, 'Source archive too large'
    for item in bundle.infolist():
        target = (SOURCE / item.filename).resolve()
        assert target.is_relative_to(SOURCE.resolve()), 'Unsafe archive path'
        assert not (item.external_attr >> 16) & 0o170000 == 0o120000, 'Symlinks are not accepted'
    bundle.extractall(SOURCE)
REPO = SOURCE / 'recursa'
assert (REPO / 'recursa/experiments.py').is_file(), 'Unexpected archive layout'
print('Python', sys.version.split()[0], '· CPU execution · no optional dependencies')

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, 'scripts/colab_bootstrap.py', '--config', CONFIG, '--output', str(OUTPUT)], cwd=REPO, check=True)
print('Durable artifacts:', OUTPUT)
print('Import a trial run.json into Recursa Studio; extract transport ZIPs first.')

## Resuming and exports
Rerun the final cell after interruption. Completed matching trials are skipped; interrupted deterministic CPU trials rerun. Failed or timed-out trials remain preserved; choose another OUTPUT directory for an explicit retry. Source/config/spec changes create a new fingerprint. Resume operates between trials, not within model/training state.
Colab resource availability and runtime lifetime vary. This notebook does not automate session creation, purchase compute, or promise unattended execution. Future authorized model runs must add verified provider-level limits and pinned dependencies first. All included measurements describe actual CPU validation only.